# Price Forecasting Models

In this notebook, we develop predictive models for Gold, Oil, and Wheat prices using ARIMA matching best practices.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error, mean_absolute_error

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

In [ ]:
conn = sqlite3.connect('../sql/commodity_data.db')
query = """
SELECT 
    p.date, 
    c.commodity_name, 
    p.price_nominal_usd
FROM prices p 
JOIN commodities c ON p.commodity_id = c.commodity_id
WHERE c.commodity_name IN ('Gold', 'Crude oil, average', 'Wheat, US SRW')
"""

df = pd.read_sql_query(query, conn)
df['date'] = pd.to_datetime(df['date'])
df.set_index('date', inplace=True)
conn.close()

# Pivot so each commodity runs in its own column
df_pivot = df.groupby(['date', 'commodity_name'])['price_nominal_usd'].mean().unstack()
df_pivot.head()

## 1. ARIMA Forecasting
Let's create an automated ARIMA trainer for our three commodities.

In [ ]:
def train_evaluate_arima(series, target_name):
    series = series.dropna()
    
    # Standard Test-train split (80-20 temporal)
    train_size = int(len(series) * 0.8)
    train, test = series.iloc[:train_size], series.iloc[train_size:]
    
    # Train basic ARIMA(5,1,0)
    model = ARIMA(train, order=(5,1,0))
    model_fit = model.fit()
    
    # Forecast
    predictions = model_fit.forecast(steps=len(test))
    
    # Evaluate
    rmse = np.sqrt(mean_squared_error(test, predictions))
    mae = mean_absolute_error(test, predictions)
    
    # Plot
    plt.figure(figsize=(12, 5))
    plt.plot(train.index, train, label='Train')
    plt.plot(test.index, test, label='Test (Actual)')
    plt.plot(test.index, predictions, color='red', label='Forecast')
    plt.title(f'{target_name} Price Forecast (RMSE: {rmse:.2f}, MAE: {mae:.2f})')
    plt.legend()
    plt.savefig(f'../images/forecast_{target_name.replace(" ", "_").lower()}.png', bbox_inches='tight')
    plt.show()
    
    return {'RMSE': rmse, 'MAE': mae}

results = {}
for col in ['Gold', 'Crude oil, average', 'Wheat, US SRW']:
    print(f"Training ARIMA for {col}...")
    results[col] = train_evaluate_arima(df_pivot[col], col)
